# UCR: ROCKET и MiniRocket vs BOSS, PatchTST

Контрольный эксперимент на тех же 8 датасетах, что и в основной таблице курсовой:
Coffee, GunPoint, Trace, TwoLeadECG, ECG5000, FordA, Wafer, ElectricDevices.

Модели: Majority, BOSS, ROCKET, MiniRocket.

Output: `ucr_rocket_results.json` + `.csv`. Runtime ≈ 20–30 мин.

In [ ]:
import sys, subprocess
def pip_install(pkgs, no_deps=False):
    cmd=[sys.executable,'-m','pip','install','--quiet']
    if no_deps: cmd.append('--no-deps')
    cmd+=pkgs
    subprocess.check_call(cmd)

pip_install(['aeon','sktime'])
print('Installed')

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
warnings.filterwarnings('ignore')

from aeon.datasets import load_classification
from aeon.classification.dictionary_based import BOSSEnsemble
from aeon.classification.convolution_based import RocketClassifier, MiniRocketClassifier

DATASETS = ['Coffee','GunPoint','Trace','TwoLeadECG','ECG5000','FordA','Wafer','ElectricDevices']
SEED = 42
results = []

In [ ]:
def evaluate(name, model, X_train, y_train, X_test, y_test, dataset):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_sec = time.time() - t0
    t0 = time.time()
    pred = model.predict(X_test)
    inf_sec = time.time() - t0
    acc = float(accuracy_score(y_test, pred))
    bal = float(balanced_accuracy_score(y_test, pred))
    f1m = float(f1_score(y_test, pred, average='macro'))
    print(f'  {name:>14} acc={acc:.4f} bal={bal:.4f} f1m={f1m:.4f} train={train_sec:.1f}s inf={inf_sec:.1f}s')
    return {'dataset':dataset,'method':name,'accuracy':acc,'balanced_accuracy':bal,'macro_f1':f1m,'train_seconds':train_sec,'infer_seconds':inf_sec}

def majority_baseline(y_train, y_test):
    cls,cnt=np.unique(y_train,return_counts=True)
    pred=np.full(len(y_test), cls[np.argmax(cnt)])
    acc=float(accuracy_score(y_test,pred))
    bal=float(balanced_accuracy_score(y_test,pred))
    f1m=float(f1_score(y_test,pred,average='macro'))
    return {'accuracy':acc,'balanced_accuracy':bal,'macro_f1':f1m}

In [ ]:
for ds in DATASETS:
    print(f'\n=== {ds} ===')
    Xtr,ytr=load_classification(ds, split='train')
    Xte,yte=load_classification(ds, split='test')
    if Xtr.ndim==3 and Xtr.shape[1]==1:
        # univariate
        pass
    n_train=Xtr.shape[0]; L=Xtr.shape[-1]; n_classes=len(np.unique(ytr))
    print(f'  train={n_train} test={Xte.shape[0]} len={L} classes={n_classes}')
    
    # Majority
    m = majority_baseline(ytr, yte)
    print(f'  {"Majority":>14} acc={m["accuracy"]:.4f} bal={m["balanced_accuracy"]:.4f} f1m={m["macro_f1"]:.4f}')
    results.append({'dataset':ds,'method':'Majority',**m,'train_seconds':0.0,'infer_seconds':0.0})
    
    # BOSS
    try:
        results.append(evaluate('BOSS', BOSSEnsemble(random_state=SEED), Xtr,ytr,Xte,yte,ds))
    except Exception as e:
        print(f'  BOSS FAIL: {e}')
    
    # ROCKET
    try:
        results.append(evaluate('ROCKET', RocketClassifier(random_state=SEED), Xtr,ytr,Xte,yte,ds))
    except Exception as e:
        print(f'  ROCKET FAIL: {e}')
    
    # MiniRocket
    try:
        results.append(evaluate('MiniRocket', MiniRocketClassifier(random_state=SEED), Xtr,ytr,Xte,yte,ds))
    except Exception as e:
        print(f'  MiniRocket FAIL: {e}')
    
    # incremental save
    with open('ucr_rocket_results.json','w') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    pd.DataFrame(results).to_csv('ucr_rocket_results.csv', index=False)

print('\nDONE. Saved ucr_rocket_results.json/.csv')

In [ ]:
df = pd.DataFrame(results)
piv = df.pivot_table(index='dataset', columns='method', values='accuracy')
print('\nAccuracy:')
print(piv.round(3))

# Mean rank (lower = better)
from scipy.stats import rankdata
ranks = piv.apply(lambda r: rankdata(-r.values), axis=1, result_type='broadcast')
print('\nMean ranks:')
print(ranks.mean(axis=0).sort_values().round(3))

from scipy.stats import friedmanchisquare
arrs = [piv[c].dropna().values for c in piv.columns]
stat, p = friedmanchisquare(*arrs)
print(f'\nFriedman: chi2={stat:.3f} p={p:.6f}')